# Hybrid Guardrail V3 — Focused Ablation Study

**Objective:** Determine which components of Hybrid Guardrail V3 contribute to performance.

**Ablations:**
- Ablation 0: Full Baseline (TF-IDF + 117 handcrafted + stacking ensemble)
- Ablation 1: TF-IDF only (no handcrafted features)
- Ablation 2: Handcrafted features only (no TF-IDF)
- Ablation 3+: Full minus one feature group at a time

**Protocol:** Same dataset, same 70/10/20 stratified split, same classifier, same seed (42).

In [12]:
import gc, re, sys, json, math, time, base64, glob, random, os, hashlib
import warnings
warnings.filterwarnings('ignore')
os.environ['LIGHTGBM_EXEC'] = 'gpu'
from pathlib import Path
from datetime import datetime
from collections import Counter
from typing import Dict, List, Optional, Set, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score,
    precision_score, recall_score, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score, precision_recall_curve,
    average_precision_score, roc_curve,
)
from scipy.sparse import hstack as sparse_hstack, csr_matrix
import xgboost as xgb
import lightgbm as lgb
import joblib

import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'psutil', 'matplotlib', 'seaborn', 'datasets'],
               capture_output=True)
import psutil
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.dpi'] = 150; plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'DejaVu Sans'; sns.set_style('whitegrid')

# GPU Detection
IN_KAGGLE = os.path.exists('/kaggle/input')
print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Local'}")

GPU_NAME = None
try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True, timeout=5)
    if r.returncode == 0:
        parts = r.stdout.strip().split(',')
        GPU_NAME = parts[0].strip()
        print(f"GPU: {GPU_NAME}")
except:
    print('GPU: detection failed')

N_CPU = os.cpu_count()
print(f"CPU cores: {N_CPU}")
ram = psutil.virtual_memory()
print(f"RAM: {ram.total/1e9:.1f}GB (avail: {ram.available/1e9:.1f}GB)")

# GPU memory management
if GPU_NAME:
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print(f"GPU memory cleared")
    except ImportError:
        pass
    try:
        subprocess.run(['nvidia-smi', '--gpu-reset', '-i=0'], capture_output=True, timeout=5)
    except:
        pass

print('All imports OK')

Environment: Kaggle
GPU: Tesla P100-PCIE-16GB
CPU cores: 4
RAM: 33.7GB (avail: 29.5GB)
GPU memory cleared
All imports OK


In [13]:
SEED = 42
TFIDF_WORD_MAX = 5000
TFIDF_NGRAM = (1, 2)
N_ESTIMATORS = 500
MAX_DEPTH = 6
LR = 0.05

# Directories
ABLATION_DIR = Path('/kaggle/working/ablation_study') if IN_KAGGLE else Path('ablation_study')
RESULTS_DIR = ABLATION_DIR / 'results'
CHECKPOINT_DIR = ABLATION_DIR / 'checkpoints'
LOG_DIR = ABLATION_DIR / 'logs'
FIG_DIR = ABLATION_DIR / 'figures'
TABLE_DIR = ABLATION_DIR / 'tables'
INTERMEDIATE_DIR = ABLATION_DIR / 'intermediate'
MANIFEST_DIR = ABLATION_DIR / 'manifests'
NOTEBOOK_DIR = ABLATION_DIR / 'notebooks'

for d in [RESULTS_DIR, CHECKPOINT_DIR, LOG_DIR, FIG_DIR, TABLE_DIR, INTERMEDIATE_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Dataset paths
if IN_KAGGLE:
    kaggle_paths = glob.glob('/kaggle/input/**/*.parquet', recursive=True)
    DATASET_PATH = kaggle_paths[0] if kaggle_paths else None
else:
    DATASET_PATH = '/home/prashanna/Documents/Guardrailer/dataset/guardrailer_dataset_v1.parquet'
    if not os.path.exists(DATASET_PATH):
        DATASET_PATH = '/kaggle/input/datasets/prashannadeveloper/guardrailer-dataset-v1/guardrailer_dataset_v1.parquet'
print(f"Dataset: {DATASET_PATH}")

np.random.seed(SEED)
print(f"Seed: {SEED}")
print(f"Ablation dir: {ABLATION_DIR}")

Dataset: /kaggle/input/datasets/prashannadeveloper/guardrailer-dataset-v1/guardrailer_dataset_v1.parquet
Seed: 42
Ablation dir: /kaggle/working/ablation_study


In [14]:
MANIFEST_PATH = MANIFEST_DIR / 'experiment_status.json'

def load_manifest() -> dict:
    if MANIFEST_PATH.exists():
        try:
            with open(MANIFEST_PATH, 'r') as f:
                return json.load(f)
        except:
            pass
    return {'experiments': {}, 'created': datetime.now().isoformat()}

def save_manifest(manifest: dict):
    tmp = MANIFEST_PATH.with_suffix('.tmp')
    with open(tmp, 'w') as f:
        json.dump(manifest, f, indent=2)
    tmp.rename(MANIFEST_PATH)

def is_experiment_completed(exp_name: str) -> bool:
    manifest = load_manifest()
    if exp_name not in manifest['experiments']:
        return False
    status = manifest['experiments'][exp_name]
    return status.get('status') == 'completed'

def update_manifest(exp_name: str, updates: dict):
    manifest = load_manifest()
    if exp_name not in manifest['experiments']:
        manifest['experiments'][exp_name] = {}
    manifest['experiments'][exp_name].update(updates)
    save_manifest(manifest)

def atomic_save_json(data: dict, path: Path):
    """Atomic write: temp file -> verify -> rename."""
    tmp = path.with_suffix('.tmp')
    with open(tmp, 'w') as f:
        json.dump(data, f, indent=2)
    # verify
    with open(tmp, 'r') as f:
        json.load(f)
    tmp.rename(path)

print('Manifest system ready')

Manifest system ready


In [15]:
ATTACK_KEYWORDS = frozenset(['ignore previous','override','bypass','jailbreak','system prompt','your instructions','forget','disregard','dan','do anything now','act as','roleplay','pretend you','hypothetical','in theory','ignore all','new instructions','you are now','developer mode','debug mode','admin mode','root mode','you must','you will','you shall','comply','obey','no restrictions','no rules','no limits','unrestricted','evil','uncensored','unfiltered','reveal','output','display','print','show','expose','previous instructions','earlier instructions','pretend','imagine','simulate','from now on','respond as','act like','system message','initial prompt'])
IMPERATIVE_VERBS = frozenset(['ignore','forget','disregard','override','bypass','reveal','show','print','output','display','act','pretend','imagine','simulate','do','let','make','enable','activate','switch','enter','execute','run','start','begin','follow','obey','comply','respond','reply','answer','tell','give','provide'])
NEGATION_WORDS = frozenset(['not','no','never','don\'t','doesn\'t','didn\'t','won\'t','wouldn\'t','can\'t','cannot','couldn\'t','shouldn\'t','mustn\'t','without','bypass','skip','remove','disable','ignore'])
MODAL_VERBS = frozenset(['must','should','shall','will','would','could','might','may','can','need','have'])
SECOND_PERSON = frozenset(['you','your','yours','yourself','yourselves'])
THIRD_PERSON = frozenset(['it','its','itself','they','them','their','theirs','he','him','his','himself','she','her','hers','herself'])
TEMPORAL = frozenset(['now','immediately','instantly','right now','from now on','henceforth','hereafter','starting now','effective immediately'])
CONDITIONAL = frozenset(['if','when','whenever','in case','assuming','provided','suppose','supposing'])
POLITENESS = frozenset(['please','kindly','if you don\'t mind','if possible','would you','could you','may i','thank you','thanks'])
URGENCY = frozenset(['urgent','important','critical','emergency','immediate','asap','right away','time sensitive'])
PERSONA_KW = frozenset(['you are','you\'re','you will be','act as','pretend to be','roleplay as','simulate being','imagine you are','assume you are','respond as'])
UNRESTRICTED_KW = frozenset(['no restrictions','no rules','no limits','no boundaries','unrestricted','uncensored','unfiltered','unlimited','without restrictions','without rules','without limits','without guidelines','without constraints','without filters'])
MODE_SWITCH_KW = frozenset(['developer mode','debug mode','admin mode','root mode','god mode','evil mode','unrestricted mode','dan mode','jailbreak mode','expert mode','master mode','override mode'])

ENCODING_PATS = {'base64':re.compile(r'[A-Za-z0-9+/]{20,}={0,2}'),'hex':re.compile(r'(?:0x[0-9a-fA-F]{2}\s*){4,}'),'url_encoded':re.compile(r'%[0-9a-fA-F]{2}'),'unicode_escape':re.compile(r'\\u[0-9a-fA-F]{4}'),'html_entity':re.compile(r'&[a-zA-Z]+;')}
STRUCTURAL_PATS = {'instruction_override':re.compile(r'(?:ignore|forget|disregard|override)\s+(?:all\s+)?(?:previous|earlier|prior|above|initial)\s+(?:instructions|rules|guidelines|prompts)'),'role_hijack':re.compile(r'(?:you\s+are\s+now|from\s+now\s+on|new\s+instructions|act\s+as\s+if)'),'system_extraction':re.compile(r'(?:reveal|show|print|output|display)\s+(?:your\s+)?(?:system\s+prompt|instructions|rules|guidelines)'),'delimiter_injection':re.compile(r'(?:```|---|\[INST\]|<<SYS>>|<\|system\|>|<\|endoftext\|>)'),'persona_switch':re.compile(r'(?:pretend|imagine|simulate|hypothetically)\s+(?:you\s+are|that\s+you|being)')}

_R_ROLE=re.compile(r'(?:USER|ASSISTANT|SYSTEM|HUMAN|AI|BOT|MODEL)\s*:',re.I)
_R_CAPS=re.compile(r'\b[A-Z]{2,}\b');_R_TRIPLE=re.compile(r'(.)\1{2,}')
_R_DELIM=re.compile(r'```|---|\[INST\]|<<SYS>>');_R_XML=re.compile(r'<[a-zA-Z]+>')
_R_BRACK=re.compile(r'[\[\]{}()]');_R_COLON=re.compile(r'(?:USER|ASSISTANT|SYSTEM|HUMAN|AI)\s*:')
_R_BULLET=re.compile(r'^\s*[-*+]\s',re.M);_R_NUM=re.compile(r'^\s*\d+\.\s',re.M)
_R_HYPH=re.compile(r'\b\w+-\w+\b');_R_ELLIP=re.compile(r'\.\.\.');_R_MD=re.compile(r'[*_`#]')
_R_LINK=re.compile(r'\[.*?\]\(.*?\)');_R_SYS=re.compile(r'\bSYSTEM\s*:',re.I)
_R_USER=re.compile(r'\bUSER\s*:',re.I);_R_ASST=re.compile(r'\bASSISTANT\s*:',re.I)
_R_NEST=re.compile(r'```[^`]*```',re.S);_R_XMLINJ=re.compile(r'<[a-zA-Z]+[^>]*>.*</[a-zA-Z]+>',re.S)
_R_COMMENTS=re.compile(r'<!--.*?-->',re.S);_R_FRAG=re.compile(r'(?:System|User|Assistant|Human|AI)\s*:',re.I)
_R_PAREN=re.compile(r'\([^)]{5,}\)');_R_INLINE=re.compile(r'`[^`]+`')
_R_HTMLTAG=re.compile(r'<[a-zA-Z][^>]*>');_R_URL=re.compile(r'https?://\S+|www\.\S+')
_R_JSONK=re.compile(r'\{[^{}]*"[^"]*"\s*:');_R_JSONA=re.compile(r'\[[^\[\]]*\{')
_R_ENC_PAYLOAD=[re.compile(p) for p in [r'(?:decode|decipher|interpret|translate)\s+(?:this|the|following)',r'(?:base64|rot13|hex|url)\s+(?:encode|decode|encoded|decoded)',r'(?:encode|decode)\s+(?:this|the|following)\s+(?:in|using|with)']]
_R_PERSONA_DEF=[re.compile(p) for p in [r'you\s+are\s+(?:a|an|the|now)',r'you\s+(?:will|shall|must|should)\s+',r'(?:never|always)\s+(?:refuse|decline|say\s+no)']]
_R_NEG_CONST=[re.compile(p) for p in [r'you\s+(?:cannot|can\'t|must\s+not|shouldn\'t|won\'t)\s+(?:refuse|decline|say\s+no)',r'(?:never|don\'t)\s+(?:refuse|decline|say\s+no)',r'no\s+(?:ethical|safety|moral)\s+(?:restrictions|limits|guidelines)']]
_R_OUTInstr=[re.compile(p) for p in [r'(?:reply|respond|answer|output|print|display|show)\s+(?:with|only|just|exactly)',r'(?:output|print|display)\s+(?:the\s+)?(?:following|below|this)',r'response\s+format']]
_R_SYSOVR=[re.compile(p) for p in [r'(?:ignore|forget|disregard|override)\s+(?:your\s+)?(?:previous|prior|earlier|all)\s+(?:instructions|rules|guidelines)',r'(?:new|updated|revised)\s+(?:instructions|rules|guidelines)\s*:',r'system\s*(?:prompt|message|instruction)\s*:']]
_HOMOGLYPH={'а':'a','е':'e','о':'о','р':'p','с':'c','у':'y','х':'x'}
_EM_DASH=frozenset({'—', '–'})
print('Keyword/pattern sets loaded')

Keyword/pattern sets loaded


In [16]:
def _sc(w):
    w=w.lower().strip()
    if len(w)<=3: return 1
    v,cnt,prev='aeiouy',0,False
    for c in w:
        iv=c in v
        if iv and not prev: cnt+=1
        prev=iv
    if w.endswith('e'): cnt-=1
    return max(1,cnt)

def _fk(ws,sc_):
    nw=len(ws)
    if nw==0 or sc_==0: return 0.0
    ns=sum(_sc(w) for w in ws)
    return 0.39*(nw/sc_)+11.8*(ns/nw)-15.59

def _cl(t,ws,sc_):
    nw=len(ws)
    if nw==0 or sc_==0: return 0.0
    nl=sum(1 for c in t if c.isalpha())
    return 0.0588*(100*nl/nw)-0.296*(100*sc_/nw)-15.8

def _nest(t):
    d=md_=0
    for c in t:
        if c in'([{':d+=1;md_=max(md_,d)
        elif c in')]}':d=max(0,d-1)
    return md_

def _md_d(t):
    if not t: return 0.0
    m=len(_R_MD.findall(t))+len(_R_LINK.findall(t))+len(_R_BULLET.findall(t))+len(_R_NUM.findall(t))
    return min(1.0,m/max(1,len(t)))

def _esc_d(t):
    if not t: return 0.0
    return min(1.0,len(re.findall(r'\\[nrtbfav\\\'"0]',t))/max(1,len(t.split())))

def _ua(t):
    if not t: return 0.0
    return min(1.0,sum(1 for c in t if c in _HOMOGLYPH)/max(1,len(t)))

def _dd(t):
    d=md_=0
    for _ in re.finditer(r'`{3,}',t):d+=1;md_=max(md_,d)
    if d>0:d=max(0,d-1)
    if d>md_:md_=d
    for _ in re.finditer(r'---+',t):
        if md_<1:md_=1
    return md_

def _ng_e(t,n):
    if len(t)<n: return 0.0
    ng=[t[i:i+n] for i in range(len(t)-n+1)]
    freq=Counter(ng);tot=len(ng)
    ent=-sum((c/tot)*math.log2(c/tot) for c in freq.values())
    me=math.log2(len(freq)) if freq else 1.0
    return ent/me if me>0 else 0.0

def _stt(ws,w):
    n=len(ws)
    if n<w: return len(set(ws))/max(1,n)
    step=w//2
    return float(np.mean([len(set(ws[i:i+w]))/w for i in range(0,n-w+1,step)]))

def _sv(ws,sc_):
    if sc_<=1 or len(ws)==0: return 0.0
    sents=re.split(r'[.!?]+',' '.join(ws))
    sents=[s.split() for s in sents if s.strip()]
    if len(sents)<=1: return 0.0
    lens=[len(s) for s in sents]
    return float(np.var(lens))

def _pd(ws,tl):
    if len(ws)<3: return 0.0
    s=0.0
    for p in _R_PERSONA_DEF:
        if p.search(tl): s+=0.5;break
    if s==0: return 0.0
    if re.search(r'you\s+(?:will|shall|must|should)\s+',tl): s+=0.3
    if re.search(r'(?:never|always)\s+(?:refuse|decline|say\s+no)',tl): s+=0.2
    return min(1.0,s)
print('Helpers loaded')

Helpers loaded


In [17]:
def extract_features(text: str) -> Dict[str, float]:
    if not text: text=' '
    tl=text.lower();words=tl.split();nw=len(words);nc=len(text);f={}
    f['char_count']=nc;f['word_count']=nw
    f['avg_word_length']=float(np.mean([len(w) for w in words])) if words else 0.0
    f['max_word_length']=float(max([len(w) for w in words])) if words else 0.0
    sc=max(1,text.count('.')+text.count('!')+text.count('?'))
    f['sentence_count']=sc;f['avg_sentence_length']=nw/sc
    f['uppercase_ratio']=sum(1 for c in text if c.isupper())/max(1,nc)
    f['digit_ratio']=sum(1 for c in text if c.isdigit())/max(1,nc)
    f['special_char_ratio']=sum(1 for c in text if not c.isalnum() and not c.isspace())/max(1,nc)
    f['space_ratio']=sum(1 for c in text if c.isspace())/max(1,nc)
    f['newline_ratio']=text.count('\n')/max(1,nc)
    f['tab_ratio']=text.count('\t')/max(1,nc)
    freq=Counter(text);tot=nc if nc else 1
    f['char_entropy']=-sum((c/tot)*math.log2(c/tot) for c in freq.values() if c>0)
    wf=Counter(words);wt=nw if nw else 1
    f['word_entropy']=-sum((c/wt)*math.log2(c/wt) for c in wf.values() if c>0)
    f['unique_word_ratio']=len(set(words))/max(1,nw)
    f['hapax_ratio']=sum(1 for c in wf.values() if c==1)/max(1,len(wf))
    kh=sum(1 for kw in ATTACK_KEYWORDS if kw in tl)
    f['attack_keyword_count']=kh;f['has_attack_keyword']=1.0 if kh>0 else 0.0
    f['keyword_density']=kh/max(1,nw)
    eh=0
    for nm,pat in ENCODING_PATS.items():
        m=pat.findall(text);f[f'encoding_{nm}']=len(m);eh+=len(m)
    f['total_encoding_hits']=eh
    sh=0
    for nm,pat in STRUCTURAL_PATS.items():
        v=1.0 if pat.search(tl) else 0.0;f[f'structural_{nm}']=v;sh+=int(v)
    f['total_structural_hits']=sh
    f['word_repeat_ratio']=1.0-f['unique_word_ratio']
    if nw>=3:
        bg=[f'{words[i]} {words[i+1]}' for i in range(nw-1)]
        f['bigram_repeat_ratio']=1.0-len(set(bg))/max(1,len(bg))
        tg=[f'{words[i]} {words[i+1]} {words[i+2]}' for i in range(nw-2)]
        f['trigram_repeat_ratio']=1.0-len(set(tg))/max(1,len(tg))
    else: f['bigram_repeat_ratio']=f['trigram_repeat_ratio']=0.0
    f['has_delimiter']=1.0 if _R_DELIM.search(text) else 0.0
    f['has_xml_tags']=1.0 if _R_XML.search(text) else 0.0
    f['has_brackets']=1.0 if _R_BRACK.search(text) else 0.0
    f['has_colon_separated']=1.0 if _R_COLON.search(text) else 0.0
    f['starts_with_imperative']=1.0 if words and words[0] in IMPERATIVE_VERBS else 0.0
    f['contains_question']=1.0 if '?' in text else 0.0
    f['exclamation_ratio']=text.count('!')/max(1,nc)
    f['triple_repeat']=1.0 if _R_TRIPLE.search(text) else 0.0
    f['word_length_variance']=float(np.var([len(w) for w in words])) if words else 0.0
    f['double_quote_count']=text.count('"');f['single_quote_count']=text.count("'")
    f['asterisk_count']=text.count('*')
    f['caps_word_count']=sum(1 for w in words if w.isupper() and len(w)>1)
    f['instruction_nesting_depth']=_nest(text)
    f['role_transition_count']=len(_R_ROLE.findall(text))
    f['markdown_density']=_md_d(text);f['delimiter_depth']=_dd(text)
    f['escape_char_density']=_esc_d(text);f['unicode_anomaly_score']=_ua(text)
    f['paragraph_count']=len(re.split(r'\n\s*\n',text.strip()))
    f['has_system_marker']=1.0 if _R_SYS.search(text) else 0.0
    f['has_user_marker']=1.0 if _R_USER.search(text) else 0.0
    f['has_assistant_marker']=1.0 if _R_ASST.search(text) else 0.0
    f['caps_sequence_count']=len(_R_CAPS.findall(text))
    f['has_inline_code']=min(1.0,len(_R_INLINE.findall(text))*0.3)
    f['has_html_tags']=min(1.0,len(_R_HTMLTAG.findall(text))*0.2)
    f['has_json_structure']=min(1.0,(0.5 if _R_JSONK.search(text) else 0)+(0.3 if _R_JSONA.search(text) else 0))
    f['has_url']=min(1.0,len(_R_URL.findall(text))*0.3)
    f['has_parenthetical']=1.0 if _R_PAREN.search(text) else 0.0
    f['imperative_verb_ratio']=sum(1 for w in words if w in IMPERATIVE_VERBS)/max(1,nw)
    f['negation_density']=sum(1 for w in words if w in NEGATION_WORDS)/max(1,nw)
    f['question_density']=text.count('?')/max(1,nw)
    mc=sum(1 for w in words if w in MODAL_VERBS);f['modal_verb_count']=mc;f['modal_verb_ratio']=mc/max(1,nw)
    spr=sum(1 for w in words if w in SECOND_PERSON)/max(1,nw);f['second_person_ratio']=spr
    f['third_person_ratio']=sum(1 for w in words if w in THIRD_PERSON)/max(1,nw)
    tj=' '.join(words)
    f['temporal_marker_count']=sum(1 for m in TEMPORAL if m in tj)
    f['conditional_marker_count']=sum(1 for w in words if w in CONDITIONAL)
    f['politeness_marker_count']=sum(1 for m in POLITENESS if m in tj)
    f['urgency_marker_count']=sum(1 for m in URGENCY if m in tj)
    f['has_second_person']=1.0 if spr>0 else 0.0
    f['has_temporal_marker']=1.0 if f['temporal_marker_count']>0 else 0.0
    f['char_trigram_entropy']=_ng_e(tl,3);f['char_quadgram_entropy']=_ng_e(tl,4)
    f['sliding_ttr_50']=_stt(words,50);f['sliding_ttr_100']=_stt(words,100)
    f['flesch_kincaid_grade']=_fk(words,sc);f['coleman_liau_index']=_cl(text,words,sc)
    f['avg_syllables']=float(np.mean([_sc(w) for w in words])) if words else 0.0
    f['max_syllables']=float(max([_sc(w) for w in words])) if words else 0.0
    f['sentence_length_variance']=_sv(words,sc)
    wl=[str(len(w)) for w in words]
    f['word_length_entropy']=_ng_e(''.join(wl),1) if wl else 0.0
    f['bigram_diversity']=len(set(' '.join(words[i:i+2]) for i in range(nw-1)))/max(1,nw-1) if nw>=2 else 0.0
    f['trigram_diversity']=len(set(' '.join(words[i:i+3]) for i in range(nw-2)))/max(1,nw-2) if nw>=3 else 0.0
    f['readability_composite']=(f['flesch_kincaid_grade']+f['coleman_liau_index'])/2.0
    f['has_encoded_payload_hint']=min(1.0,sum(1 for p in _R_ENC_PAYLOAD if p.search(tl))*0.4)
    f['encoding_instruction_ratio']=f['has_encoded_payload_hint']
    f['has_persona_definition']=_pd(words,tl)
    f['has_negative_constraints']=min(1.0,sum(1 for p in _R_NEG_CONST if p.search(tl))*0.5)
    f['has_output_instruction']=min(1.0,sum(1 for p in _R_OUTInstr if p.search(tl))*0.4)
    f['has_system_override']=min(1.0,sum(1 for p in _R_SYSOVR if p.search(tl))*0.4)
    f['persona_keyword_count']=sum(1 for pk in PERSONA_KW if pk in tl)
    f['unrestricted_keyword_count']=sum(1 for uk in UNRESTRICTED_KW if uk in tl)
    f['mode_switch_count']=sum(1 for mk in MODE_SWITCH_KW if mk in tl)
    f['has_nested_delimiters']=1.0 if _R_NEST.search(text) else 0.0
    f['has_xml_injection']=1.0 if _R_XMLINJ.search(text) else 0.0
    f['has_comment_injection']=1.0 if _R_COMMENTS.search(text) else 0.0
    f['has_prompt_fragment']=1.0 if _R_FRAG.search(text) else 0.0
    f['colon_count']=text.count(':');f['semicolon_count']=text.count(';')
    f['pipe_count']=text.count('|')
    f['angle_bracket_count']=text.count('<')+text.count('>')
    f['curly_brace_count']=text.count('{')+text.count('}')
    f['square_bracket_count']=text.count('[')+text.count(']')
    f['backtick_count']=text.count('`');f['tilde_count']=text.count('~')
    f['hyphen_sequence_count']=len(re.findall(r'-{3,}',text))
    f['underscore_sequence_count']=len(re.findall(r'_{3,}',text))
    f['has_hyphenated_compound']=1.0 if _R_HYPH.search(text) else 0.0
    f['has_ellipsis']=1.0 if _R_ELLIP.search(text) else 0.0
    f['has_em_dash']=1.0 if any(c in _EM_DASH for c in text) else 0.0
    f['has_bullet_list']=1.0 if _R_BULLET.search(text) else 0.0
    f['has_numbered_list']=1.0 if _R_NUM.search(text) else 0.0
    return f

def get_feature_names():
    return sorted(extract_features('test').keys())

def extract_features_batch(texts, n_workers=None):
    """Extract features in parallel using multiprocessing."""
    if n_workers is None:
        n_workers = min(os.cpu_count() or 4, 8)
    if len(texts) < 1000 or n_workers <= 1:
        # Small batch: sequential is faster than mp overhead
        af = [extract_features(t) for t in texts]
        names = sorted(af[0].keys())
        return np.array([[fd[k] for k in names] for fd in af], dtype=np.float32), names
    # Parallel extraction
    from concurrent.futures import ProcessPoolExecutor, as_completed
    chunk_size = max(100, len(texts) // (n_workers * 4))
    chunks = [texts[i:i+chunk_size] for i in range(0, len(texts), chunk_size)]
    results_dict = {}
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(_extract_chunk, chunk): idx for idx, chunk in enumerate(chunks)}
        for future in as_completed(futures):
            idx = futures[future]
            results_dict[idx] = future.result()
    # Merge in order
    all_dicts = [results_dict[i] for i in range(len(chunks))]
    names = sorted(all_dicts[0][0].keys())
    rows = []
    for d in all_dicts:
        rows.extend([[fd[k] for k in names] for fd in d])
    return np.array(rows, dtype=np.float32), names

def _extract_chunk(texts_chunk):
    return [extract_features(t) for t in texts_chunk]

# =============================================================================
# FEATURE GROUPS — organized by semantic category
# =============================================================================
FEATURE_GROUPS = {
    'text_statistics': [
        'avg_sentence_length', 'avg_word_length', 'char_count', 'max_word_length',
        'sentence_count', 'word_count',
    ],
    'char_composition': [
        'digit_ratio', 'newline_ratio', 'space_ratio', 'special_char_ratio',
        'tab_ratio', 'uppercase_ratio',
    ],
    'char_counts': [
        'angle_bracket_count', 'asterisk_count', 'backtick_count', 'colon_count',
        'curly_brace_count', 'double_quote_count', 'pipe_count', 'semicolon_count',
        'single_quote_count', 'square_bracket_count', 'tilde_count',
        'hyphen_sequence_count', 'underscore_sequence_count',
    ],
    'entropy_diversity': [
        'char_entropy', 'char_quadgram_entropy', 'char_trigram_entropy',
        'word_entropy', 'word_length_entropy',
        'bigram_diversity', 'bigram_repeat_ratio', 'hapax_ratio',
        'trigram_diversity', 'trigram_repeat_ratio',
        'unique_word_ratio', 'word_repeat_ratio',
        'sliding_ttr_50', 'sliding_ttr_100',
    ],
    'keywords': [
        'attack_keyword_count', 'has_attack_keyword', 'keyword_density',
    ],
    'encoding_patterns': [
        'encoding_base64', 'encoding_hex', 'encoding_html_entity',
        'total_encoding_hits', 'encoding_unicode_escape', 'encoding_url_encoded',
    ],
    'structural_patterns': [
        'structural_delimiter_injection', 'structural_instruction_override',
        'structural_persona_switch', 'structural_role_hijack',
        'structural_system_extraction', 'total_structural_hits',
    ],
    'text_booleans': [
        'contains_question', 'has_brackets', 'has_colon_separated',
        'has_delimiter', 'has_ellipsis', 'has_em_dash',
        'has_parenthetical', 'has_xml_tags', 'triple_repeat',
        'has_bullet_list', 'has_hyphenated_compound', 'has_numbered_list',
        'starts_with_imperative',
    ],
    'code_markup': [
        'caps_sequence_count', 'caps_word_count',
        'has_inline_code', 'has_html_tags', 'has_json_structure',
        'has_url',
    ],
    'markers_roles': [
        'has_assistant_marker', 'has_system_marker', 'has_user_marker',
        'role_transition_count',
    ],
    'nesting_structure': [
        'delimiter_depth', 'instruction_nesting_depth', 'markdown_density',
        'paragraph_count',
    ],
    'unicode_escape': [
        'escape_char_density', 'unicode_anomaly_score',
    ],
    'linguistic': [
        'conditional_marker_count', 'imperative_verb_ratio',
        'modal_verb_count', 'modal_verb_ratio', 'negation_density',
        'politeness_marker_count', 'question_density',
        'second_person_ratio', 'temporal_marker_count',
        'third_person_ratio', 'urgency_marker_count',
        'has_second_person', 'has_temporal_marker',
    ],
    'attack_specific': [
        'has_encoded_payload_hint', 'encoding_instruction_ratio',
        'has_negative_constraints', 'has_output_instruction',
        'has_system_override', 'persona_keyword_count',
        'unrestricted_keyword_count', 'mode_switch_count',
        'has_persona_definition',
    ],
    'injection_detection': [
        'has_nested_delimiters', 'has_xml_injection',
        'has_comment_injection', 'has_prompt_fragment',
    ],
    'readability': [
        'coleman_liau_index', 'flesch_kincaid_grade', 'readability_composite',
        'avg_syllables', 'max_syllables', 'word_length_variance',
        'sentence_length_variance',
    ],
    'punctuation_misc': [
        'exclamation_ratio',
    ],
}

NF = len(get_feature_names())
total_grouped = sum(len(v) for v in FEATURE_GROUPS.values())
print(f'Total features: {NF}')
print(f'Grouped features: {total_grouped}')
print(f'Feature groups ({len(FEATURE_GROUPS)}):')
for gname, gfeats in FEATURE_GROUPS.items():
    print(f'  {gname}: {len(gfeats)} features')
assert total_grouped == NF, f'Mismatch: {total_grouped} grouped vs {NF} total'

Total features: 117
Grouped features: 117
Feature groups (17):
  text_statistics: 6 features
  char_composition: 6 features
  char_counts: 13 features
  entropy_diversity: 14 features
  keywords: 3 features
  encoding_patterns: 6 features
  structural_patterns: 6 features
  text_booleans: 13 features
  code_markup: 6 features
  markers_roles: 4 features
  nesting_structure: 4 features
  unicode_escape: 2 features
  linguistic: 13 features
  attack_specific: 9 features
  injection_detection: 4 features
  readability: 7 features
  punctuation_misc: 1 features


In [18]:
class RegexPatternScorer:
    def __init__(self, pat_dict):
        self.patterns=[]
        for pat,score in pat_dict.items():
            try:
                self.patterns.append((re.compile(pat,re.I),score))
            except re.error:
                self.patterns.append((re.compile(re.escape(pat),re.I),score))
    def max_score(self,text):
        mx=0.0
        for pat,score in self.patterns:
            if pat.search(text):
                if score>mx: mx=score
        return mx

pats={'ignore\\s+(?:all\\s+)?(?:previous|earlier|prior|above|initial)\\s+(?:instructions|rules|guidelines|prompts)':0.95,'forget\\s+(?:all\\s+)?(?:previous|earlier|prior|above)\\s+(?:instructions|rules|context)':0.95,'disregard\\s+(?:all\\s+)?(?:previous|earlier|prior|above)\\s+(?:instructions|rules)':0.95,'override\\s+(?:all\\s+)?(?:previous|earlier|prior|safety)\\s+(?:instructions|rules|guidelines)':0.95,'you\\s+are\\s+now\\s+(?:a|an|the|called|named)':0.9,'from\\s+now\\s+on':0.9,'new\\s+instructions?[\\s:]+':0.85,'(?:enter|switch\\s+to|activate)\\s+(?:developer|debug|admin|root|god|evil|unrestricted)\\s+mode':0.95,'(?:do|be)\\s+anything\\s+now':0.9,'no\\s+(?:restrictions|rules|limits|boundaries|guidelines)':0.9,'dan\\s+(?:mode|version|prompt)':0.95,'(?:bypass|skip|remove|disable)\\s+(?:all\\s+)?(?:safety|content|security|moderation)':0.95,'(?:reveal|show|print|output|display)\\s+(?:your\\s+)?(?:system\\s+prompt|instructions)':0.95,'(?:repeat|echo)\\s+(?:your\\s+)?(?:system|initial)\\s+(?:prompt|instructions)':0.9,'(?:pretend|imagine|simulate)\\s+(?:you(?:\'|\\s+are)\\s+)?(?:a|an|the|being)?':0.7,'(?:hypothetical(?:ly)?|in\\s+theory)[\\s,]+':0.6,'act\\s+as\\s+(?:if|though)':0.8,'without\\s+(?:any\\s+)?(?:restrictions|rules|limits|guidelines)':0.9,'(?:unrestricted|uncensored|unfiltered)\\s+(?:mode|version|model)':0.9,'(?:you\\s+must|you\\s+will|you\\s+shall)\\s+(?:comply|obey|follow)':0.85,'(?:never|don\'?t)\\s+(?:refuse|decline|say\\s+no)':0.85,'(?:no|without)\\s+(?:ethical|moral|safety)\\s+(?:constraints|restrictions|guidelines)':0.9,'`{3,}\\s*\\n.*(?:system|prompt|instructions)':0.85,'\\[INST\\]|<<SYS>>|<\\|im_start\\|>':0.9,'(?:USER|ASSISTANT|SYSTEM|HUMAN|AI)\\s*:':0.6,'(?:drop|remove|disable)\\s+(?:your\\s+)?(?:filters|restrictions|guardrails)':0.9,'(?:give|grant|elevate)\\s+(?:me\\s+)?(?:admin|root|god| elevated)\\s+(?:access|privileges)':0.9,'you\\s+are\\s+(?:now\\s+)?(?:a\\s+)?(?:hacker|attacker|red\\s+team)':0.85,'(?:respond|reply|answer)\\s+(?:without|no|bypassing)\\s+(?:filters|restrictions|moderation)':0.9,'(?:write|create|generate)\\s+(?:a\\s+)?(?:malicious|harmful|dangerous|exploit)':0.85,'(?:how\\s+to|method\\s+to|ways\\s+to)\\s+(?:hack|exploit|bypass|attack)':0.8,'(?:ignore|override|bypass)\\s+(?:all\\s+)?(?:content\\s+)?(?:policy|policies|guidelines)':0.95,'(?:you\\s+are\\s+now\\s+)?(?:DAN|jailbroken|unlocked)':0.9,'(?:enter|enable|activate)\\s+(?:DAN|jailbreak|developer)\\s+mode':0.95,'(?:previous|earlier|all)\\s+instructions\\s+are\\s+(?:cancelled|void|overridden)':0.95,'(?:I\\s+need|I\\s+want|give\\s+me)\\s+(?:you\\s+to\\s+)?(?:bypass|override|ignore)':0.85,'(?:forget|disregard|ignore)\\s+everything':0.9,'(?:system\\s+)?(?:prompt|message)\\s*(?:override|override|takeover)':0.9,'(?:now\\s+you\\s+are|from\\s+now\\s+you\\s+will|new\\s+role)':0.85,'(?:respond|act|behave)\\s+as\\s+if\\s+you\\s+(?:have|had|are)\\s+no':0.85,'(?:no\\s+more\\s+)?(?:safety|content)\\s+(?:filters|restrictions|guidelines|protocols)':0.9,'(?:tell|show|reveal|expose)\\s+me\\s+(?:your\\s+)?(?:system\\s+)?(?:prompt|instructions)':0.9,'(?:repeat|print|output)\\s+(?:the\\s+)?(?:above|previous|initial)\\s+(?:text|prompt|instructions)':0.9,'(?:continue|repeat)\\s+(?:the\\s+)?(?:conversation|text|prompt)\\s+(?:without|no)':0.85}

aho=RegexPatternScorer(pats)
print(f'RegexPatternScorer: {len(aho.patterns)} patterns')

class TFIDFSimilarity:
    def __init__(self,wf=20000,cf=10000):
        self.wf=wf;self.cf=cf;self.vec=self.cvec=None;self.mc_int8=self.cmc_int8=None;self.ok=False
    def fit(self,texts,labels=None):
        print(f'  Word TF-IDF ({self.wf})...')
        self.vec=TfidfVectorizer(max_features=self.wf,sublinear_tf=True,norm='l2',ngram_range=(1,2),dtype=np.float32)
        X=self.vec.fit_transform(texts)
        m=None
        if labels is not None:
            la=np.array(labels);m=la==1
            if m.sum()>0:
                mc=np.asarray(X[m].mean(axis=0)).flatten().astype(np.float32)
                cn=np.linalg.norm(mc)
                if cn>0: mc/=cn
                self.mc_int8=(mc*127).astype(np.int8)
        del X;gc.collect()
        print(f'  Char TF-IDF ({self.cf})...')
        self.cvec=TfidfVectorizer(max_features=self.cf,analyzer='char_wb',ngram_range=(3,5),sublinear_tf=True,norm='l2',dtype=np.float32)
        Xc=self.cvec.fit_transform(texts)
        if m is not None and m.sum()>0:
            cmc=np.asarray(Xc[m].mean(axis=0)).flatten().astype(np.float32)
            cn=np.linalg.norm(cmc)
            if cn>0: cmc/=cn
            self.cmc_int8=(cmc*127).astype(np.int8)
        del Xc;gc.collect();self.ok=True
    def _sim_q(self,X,c8):
        if X.nnz==0: return 0.0
        d=np.asarray(X.todense()).flatten().astype(np.float32)
        n=np.linalg.norm(d)
        if n==0: return 0.0
        d_norm=d/n
        q=(d_norm*127).astype(np.int8)
        dot=float(np.dot(q,c8))
        return 1.0/(1.0+np.exp(-5.0*(dot/127-0.3)))
    def score(self,text):
        ws=self._sim_q(self.vec.transform([text]),self.mc_int8) if self.ok and self.mc_int8 is not None else 0.0
        cs=self._sim_q(self.cvec.transform([text]),self.cmc_int8) if self.ok and self.cmc_int8 is not None else 0.0
        return max(ws,cs)
    def save(self,p):joblib.dump({'vec':self.vec,'cvec':self.cvec,'mc_int8':self.mc_int8,'cmc_int8':self.cmc_int8,'wf':self.wf,'cf':self.cf},p)
    def load(self,p):d=joblib.load(p);self.vec=d['vec'];self.cvec=d['cvec'];self.mc_int8=d['mc_int8'];self.cmc_int8=d['cmc_int8'];self.ok=True

class IDFKeywords:
    def __init__(self):self.kw={};self.ok=False
    def fit(self,texts,labels):
        n=len(texts);md,sd=Counter(),Counter()
        for t,l in zip(texts,labels):
            s=set(t.lower().split())
            for w in s: (md if l==1 else sd)[w]+=1
        for w,doc_freq in md.items():
            if doc_freq>=3: idf=np.log((n-doc_freq+0.5)/(doc_freq+0.5)+1.0);self.kw[w]=idf*doc_freq/(doc_freq+sd.get(w,0)+1)
        self.ok=True
    def score(self,text):
        if not self.ok: return 0.0
        ws=text.lower().split()
        if not ws: return 0.0
        return float(1.0/(1.0+np.exp(-10.0*(sum(self.kw.get(w,0) for w in ws)/len(ws)-0.1))))
    def save(self,p):joblib.dump(self.kw,p)
    def load(self,p):self.kw=joblib.load(p);self.ok=True

print('Similarity + Keywords loaded')

RegexPatternScorer: 44 patterns
Similarity + Keywords loaded


In [19]:
print('Loading full dataset...')
df = pd.read_parquet(DATASET_PATH, columns=['prompt_text', 'is_malicious'])
df['is_malicious'] = df['is_malicious'].astype(int)
print(f'Dataset: {len(df):,} rows')
if len(df) == 0:
    raise ValueError(f'Dataset is empty: {DATASET_PATH}')
vc = df['is_malicious'].value_counts()
if len(vc) < 2:
    raise ValueError(f'Dataset has only one class: {vc.to_dict()}')
print(f'Class distribution: {vc.to_dict()}')

texts = df['prompt_text'].astype(str).tolist()
labels = df['is_malicious'].values.astype(int)
del df; gc.collect()

Xtr, Xtmp, ytr, ytmp = train_test_split(texts, labels, test_size=0.2, random_state=SEED, stratify=labels)
Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.5, random_state=SEED, stratify=ytmp)
del texts, labels, Xtmp, ytmp; gc.collect()

print(f'Train: {len(Xtr):,} ({sum(ytr):,} malicious)')
print(f'Val:   {len(Xva):,} ({sum(yva):,} malicious)')
print(f'Test:  {len(Xte):,} ({sum(yte):,} malicious)')
print('Data loaded. Split: 80/10/10 of original = 70/10/20 of total.')
print('Test set is FIXED and identical across all ablations.')

Loading full dataset...
Dataset: 722,842 rows
Class distribution: {1: 433066, 0: 289776}
Train: 578,273 (346,452 malicious)
Val:   72,284 (43,307 malicious)
Test:  72,285 (43,307 malicious)
Data loaded. Split: 80/10/10 of original = 70/10/20 of total.
Test set is FIXED and identical across all ablations.


In [20]:
import multiprocessing as mp

# Global feature cache - computed ONCE, reused across all experiments
_FEATURE_CACHE = {
    'all_features': None,      # Full 117-feature matrix for train/val/test
    'feature_names': None,     # Sorted feature names
    'tfidf_full': None,        # TF-IDF vectorizer fitted on train
    'tfidf_train': None,       # TF-IDF train matrix
    'tfidf_val': None,         # TF-IDF val matrix
    'tfidf_test': None,        # TF-IDF test matrix
}

def precompute_all_features(Xtr_texts, Xva_texts, Xte_texts, n_workers=None):
    """Precompute ALL 117 handcrafted features ONCE for all splits."""
    if _FEATURE_CACHE['all_features'] is not None:
        print('  Features already cached, skipping...')
        return
    
    if n_workers is None:
        n_workers = min(os.cpu_count() or 4, 8)
    
    total = len(Xtr_texts) + len(Xva_texts) + len(Xte_texts)
    print(f'  Precomputing 117 features for {total:,} texts using {n_workers} workers...')
    t0 = time.time()
    
    # Combine all texts for single-pass extraction
    all_texts = Xtr_texts + Xva_texts + Xte_texts
    
    # Parallel extraction in chunks
    from concurrent.futures import ProcessPoolExecutor, as_completed
    chunk_size = max(500, len(all_texts) // (n_workers * 4))
    chunks = [all_texts[i:i+chunk_size] for i in range(0, len(all_texts), chunk_size)]
    
    all_dicts = [None] * len(chunks)
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(_extract_chunk, chunk): idx for idx, chunk in enumerate(chunks)}
        done = 0
        for future in as_completed(futures):
            idx = futures[future]
            all_dicts[idx] = future.result()
            done += 1
            if done % 10 == 0 or done == len(chunks):
                pct = done / len(chunks) * 100
                elapsed = time.time() - t0
                eta = elapsed / done * (len(chunks) - done) if done > 0 else 0
                print(f'    [{done}/{len(chunks)}] {pct:.0f}% ({elapsed:.1f}s, ETA {eta:.1f}s)')
    
    # Merge results
    names = sorted(all_dicts[0][0].keys())
    rows = []
    for d in all_dicts:
        rows.extend([[fd[k] for k in names] for fd in d])
    
    all_features = np.array(rows, dtype=np.float32)
    n_train = len(Xtr_texts)
    n_val = len(Xva_texts)
    n_test = len(Xte_texts)
    
    _FEATURE_CACHE['all_features'] = {
        'train': all_features[:n_train],
        'val': all_features[n_train:n_train+n_val],
        'test': all_features[n_train+n_val:],
    }
    _FEATURE_CACHE['feature_names'] = names
    
    elapsed = time.time() - t0
    print(f'  Features precomputed: {all_features.shape[1]} features, {total:,} samples in {elapsed:.1f}s')
    print(f'  Speed: {total/elapsed:.0f} samples/sec')

def precompute_tfidf(Xtr_texts, Xva_texts, Xte_texts):
    """Precompute TF-IDF ONCE for all experiments that use it."""
    if _FEATURE_CACHE['tfidf_full'] is not None:
        print('  TF-IDF already cached, skipping...')
        return
    
    print(f'  Fitting TF-IDF on {len(Xtr_texts):,} training texts...')
    t0 = time.time()
    
    tfidf = TfidfVectorizer(max_features=TFIDF_WORD_MAX, sublinear_tf=True,
                            norm='l2', ngram_range=TFIDF_NGRAM, dtype=np.float32)
    Xt_train = tfidf.fit_transform(Xtr_texts)
    Xt_val = tfidf.transform(Xva_texts)
    Xt_test = tfidf.transform(Xte_texts)
    
    _FEATURE_CACHE['tfidf_full'] = tfidf
    _FEATURE_CACHE['tfidf_train'] = Xt_train
    _FEATURE_CACHE['tfidf_val'] = Xt_val
    _FEATURE_CACHE['tfidf_test'] = Xt_test
    
    elapsed = time.time() - t0
    print(f'  TF-IDF fitted: {Xt_train.shape[1]} features in {elapsed:.1f}s')

def get_cached_features(split, feature_mask=None):
    """Get cached handcrafted features for a split, optionally masked."""
    feats = _FEATURE_CACHE['all_features'][split]
    if feature_mask and len(feature_mask) < feats.shape[1]:
        return feats[:, feature_mask]
    return feats

def get_cached_tfidf(split):
    """Get cached TF-IDF matrix for a split."""
    key = f'tfidf_{split}'
    return _FEATURE_CACHE[key]

def clear_feature_cache():
    """Free memory from feature cache."""
    for k in _FEATURE_CACHE:
        _FEATURE_CACHE[k] = None
    gc.collect()

print('Feature cache system ready')

Feature cache system ready


In [21]:
class ExperimentRunner:
    """Runs a single ablation experiment using precomputed features."""
    
    def __init__(self, name, description,
                 use_tfidf=True, use_handcrafted=True,
                 exclude_groups=None, include_groups=None):
        self.name = name
        self.description = description
        self.use_tfidf = use_tfidf
        self.use_handcrafted = use_handcrafted
        self.exclude_groups = exclude_groups or []
        self.include_groups = include_groups
    
    def _get_feature_mask(self, feature_names):
        if not self.use_handcrafted:
            return []
        if self.include_groups is not None:
            allowed = set()
            for g in self.include_groups:
                allowed.update(FEATURE_GROUPS[g])
            return [i for i, n in enumerate(feature_names) if n in allowed]
        if self.exclude_groups:
            excluded = set()
            for g in self.exclude_groups:
                excluded.update(FEATURE_GROUPS[g])
            return [i for i, n in enumerate(feature_names) if n not in excluded]
        return list(range(len(feature_names)))
    
    def run(self, Xtr_texts, ytr, Xva_texts, yva, Xte_texts, yte,
            feature_names, SEED):
        result = {
            'name': self.name, 'description': self.description,
            'use_tfidf': self.use_tfidf, 'use_handcrafted': self.use_handcrafted,
            'exclude_groups': self.exclude_groups, 'include_groups': self.include_groups,
            'train_samples': len(Xtr_texts), 'val_samples': len(Xva_texts),
            'test_samples': len(Xte_texts), 'random_seed': SEED,
            'start_time': datetime.now().isoformat(),
        }
        
        feature_mask = self._get_feature_mask(feature_names)
        result['n_handcrafted_features'] = len(feature_mask)
        result['tfidf_max_features'] = TFIDF_WORD_MAX if self.use_tfidf else 0
        result['tfidf_ngram'] = list(TFIDF_NGRAM) if self.use_tfidf else []
        
        # Build combined feature matrix from CACHE (instant)
        t0 = time.time()
        
        if self.use_tfidf:
            Xt_train = get_cached_tfidf('train')
            Xt_val = get_cached_tfidf('val')
            Xt_test = get_cached_tfidf('test')
            result['tfidf_vocab_size'] = len(_FEATURE_CACHE['tfidf_full'].vocabulary_)
        else:
            Xt_train = csr_matrix((len(Xtr_texts), 0), dtype=np.float32)
            Xt_val = csr_matrix((len(Xva_texts), 0), dtype=np.float32)
            Xt_test = csr_matrix((len(Xte_texts), 0), dtype=np.float32)
            result['tfidf_vocab_size'] = 0
        
        if self.use_handcrafted:
            Xh_train = get_cached_features('train', feature_mask)
            Xh_val = get_cached_features('val', feature_mask)
            Xh_test = get_cached_features('test', feature_mask)
        else:
            n_feat = 0
            Xh_train = csr_matrix((len(Xtr_texts), n_feat), dtype=np.float32)
            Xh_val = csr_matrix((len(Xva_texts), n_feat), dtype=np.float32)
            Xh_test = csr_matrix((len(Xte_texts), n_feat), dtype=np.float32)
        
        Xtr_c = sparse_hstack([csr_matrix(Xh_train, dtype=np.float32), Xt_train], format='csr')
        Xva_c = sparse_hstack([csr_matrix(Xh_val, dtype=np.float32), Xt_val], format='csr')
        Xte_c = sparse_hstack([csr_matrix(Xh_test, dtype=np.float32), Xt_test], format='csr')
        
        del Xh_train, Xh_val, Xh_test
        gc.collect()
        
        result['total_features'] = Xtr_c.shape[1]
        result['feature_build_time'] = time.time() - t0
        print(f'    Features: {Xtr_c.shape[1]} ({len(feature_mask)} HC + {result["tfidf_vocab_size"]} TFIDF) in {result["feature_build_time"]:.3f}s')
        
        # Train stacking ensemble - GPU optimized
        print(f'  Training stacking ensemble...')
        t0 = time.time()
        
        ests = []
        _xgb_cpu = xgb.XGBClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                                      learning_rate=LR, subsample=0.8, colsample_bytree=0.8,
                                      min_child_weight=5, gamma=0.5, reg_alpha=1.0, reg_lambda=1.0,
                                      eval_metric='logloss', tree_method='hist',
                                      random_state=SEED, n_jobs=-1)
        try:
            _xgb_gpu = xgb.XGBClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                                          learning_rate=LR, subsample=0.8, colsample_bytree=0.8,
                                          min_child_weight=5, gamma=0.5, reg_alpha=1.0, reg_lambda=1.0,
                                          eval_metric='logloss', tree_method='hist',
                                          random_state=SEED, n_jobs=-1, device='cuda')
            _xgb_gpu.fit(np.zeros((1,1)), [0])
            del _xgb_gpu
            ests.append(('xgb', _xgb_cpu.set_params(device='cuda')))
            print('    XGBoost: GPU')
        except Exception:
            ests.append(('xgb', _xgb_cpu))
            print('    XGBoost: CPU')
        
        _lgb_cpu = lgb.LGBMClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                                      learning_rate=LR, subsample=0.8, colsample_bytree=0.8,
                                      min_child_weight=5, reg_alpha=1.0, reg_lambda=1.0,
                                      objective='binary', metric='binary_logloss',
                                      random_state=SEED, n_jobs=-1, verbose=-1)
        try:
            _lgb_gpu = lgb.LGBMClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                                          learning_rate=LR, subsample=0.8, colsample_bytree=0.8,
                                          min_child_weight=5, reg_alpha=1.0, reg_lambda=1.0,
                                          objective='binary', metric='binary_logloss',
                                          random_state=SEED, n_jobs=-1, verbose=-1,
                                          device='gpu', gpu_use_dp=False, platform_config='OpenCL')
            _lgb_gpu.fit(np.zeros((1,1)), [0])
            del _lgb_gpu
            _lgb_cpu.set_params(device='gpu', gpu_use_dp=False, platform_config='OpenCL')
            ests.append(('lgbm', _lgb_cpu))
            print('    LightGBM: GPU')
        except Exception:
            ests.append(('lgbm', _lgb_cpu))
            print('    LightGBM: CPU')
        
        _rf = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_split=5,
                                      min_samples_leaf=2, max_features='sqrt',
                                      random_state=SEED, n_jobs=-1)
        ests.append(('rf', _rf))
        print('    RandomForest: CPU (all cores)')
        
        stk = StackingClassifier(
            estimators=ests,
            final_estimator=LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
            cv=3, stack_method='predict_proba', n_jobs=3, passthrough=True)
        stk.fit(Xtr_c, ytr)
        
        train_score = stk.score(Xtr_c, ytr)
        val_score = stk.score(Xva_c, yva)
        result['train_time'] = time.time() - t0
        result['train_accuracy'] = float(train_score)
        result['val_accuracy'] = float(val_score)
        print(f'    Train: {train_score:.4f}, Val: {val_score:.4f} ({result["train_time"]:.1f}s)')
        del Xtr_c
        gc.collect()
        
        # Calibration + threshold
        print(f'  Calibrating...')
        t0 = time.time()
        val_idx = np.arange(len(yva))
        skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED)
        tr_i, th_i = next(skf.split(val_idx, yva))
        Xcal_tr = Xva_c[tr_i]; ycal_tr = yva[tr_i]
        Xcal_th = Xva_c[th_i]; ycal_th = yva[th_i]
        
        cal = CalibratedClassifierCV(stk, method='isotonic', cv=3)
        cal.fit(Xcal_tr, ycal_tr)
        
        # Batch predict on calibration threshold half
        vp = cal.predict_proba(Xcal_th)[:, 1]
        
        def ece(yt, yp, nb=10):
            edges = np.linspace(0, 1, nb+1); e = 0.0
            for i in range(nb):
                mask = (yp >= edges[i]) & (yp < edges[i+1])
                if mask.sum() > 0:
                    e += mask.sum()/len(yt) * abs(yt[mask].mean() - yp[mask].mean())
            return float(e)
        
        ece_val = ece(ycal_th, vp)
        
        bt, bf = 0.5, 0.0
        for t in np.arange(0.05, 0.95, 0.01):
            fi = f1_score(ycal_th, (vp >= t).astype(int), zero_division=0)
            if fi > bf:
                bf = fi; bt = t
        thr = bt
        result['ece'] = float(ece_val)
        result['threshold'] = float(thr)
        result['calibration_f1'] = float(bf)
        result['calibration_time'] = time.time() - t0
        print(f'    ECE: {ece_val:.4f}, Threshold: {thr:.2f} ({result["calibration_time"]:.1f}s)')
        del Xcal_tr, ycal_tr, Xcal_th, ycal_th, vp
        gc.collect()
        
        # Test evaluation - batch prediction
        print(f'  Evaluating on test set...')
        t0 = time.time()
        batch_size = 10000
        tp_parts = []
        for i in range(0, len(yte), batch_size):
            batch = Xte_c[i:i+batch_size]
            tp_parts.append(cal.predict_proba(batch)[:, 1])
        tp_ = np.concatenate(tp_parts)
        yp = (tp_ >= thr).astype(int)
        it = time.time() - t0
        
        nt = len(yte)
        result['inference_time'] = float(it)
        result['inference_latency_ms'] = float(it / nt * 1000)
        result['inference_throughput'] = float(nt / it)
        print(f'    Inference: {it:.2f}s ({nt/it:.0f}/s, {it/nt*1000:.3f}ms/prompt)')
        
        # Metrics
        tn, fp, fn, tp = confusion_matrix(yte, yp).ravel()
        ac = accuracy_score(yte, yp)
        ba = balanced_accuracy_score(yte, yp)
        pr = precision_score(yte, yp, zero_division=0)
        rc = recall_score(yte, yp, zero_division=0)
        f1_ = f1_score(yte, yp, zero_division=0)
        mc = matthews_corrcoef(yte, yp)
        ka = cohen_kappa_score(yte, yp)
        try: au = roc_auc_score(yte, tp_)
        except: au = None
        try: ap = average_precision_score(yte, tp_)
        except: ap = None
        
        fpr_val = fp / max(1, fp + tn)
        fnr_val = fn / max(1, fn + tp)
        
        result['accuracy'] = float(ac)
        result['balanced_accuracy'] = float(ba)
        result['precision'] = float(pr)
        result['recall'] = float(rc)
        result['f1'] = float(f1_)
        result['mcc'] = float(mc)
        result['kappa'] = float(ka)
        result['roc_auc'] = float(au) if au is not None else None
        result['pr_auc'] = float(ap) if ap is not None else None
        result['fpr'] = float(fpr_val)
        result['fnr'] = float(fnr_val)
        result['tp'] = int(tp); result['tn'] = int(tn)
        result['fp'] = int(fp); result['fn'] = int(fn)
        
        print(f'\n    === {self.name} ===')
        print(classification_report(yte, yp, target_names=['Safe', 'Malicious'], zero_division=0))
        print(f'    Acc={ac:.4f} F1={f1_:.4f} AUC={au:.4f if au else "N/A"} FPR={fpr_val:.4f} FNR={fnr_val:.4f}')
        
        # Save results
        cm_ = confusion_matrix(yte, yp)
        atomic_save_json({'confusion_matrix': cm_.tolist(), 'labels': ['Safe', 'Malicious']},
                        RESULTS_DIR / f'{self.name}_confusion.json')
        if au is not None:
            fpr_curve, tpr_curve, _ = roc_curve(yte, tp_)
            atomic_save_json({'fpr': fpr_curve.tolist(), 'tpr': tpr_curve.tolist(), 'auc': au},
                            RESULTS_DIR / f'{self.name}_roc.json')
        if ap is not None:
            prec_curve, rec_curve, _ = precision_recall_curve(yte, tp_)
            atomic_save_json({'precision': prec_curve.tolist(), 'recall': rec_curve.tolist(), 'ap': ap},
                            RESULTS_DIR / f'{self.name}_pr.json')
        atomic_save_json(result, RESULTS_DIR / f'{self.name}_result.json')
        
        # Cleanup
        del Xtr_c, Xte_c, stk, cal, tp_, yp
        gc.collect()
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except: pass
        
        result['end_time'] = datetime.now().isoformat()
        result['status'] = 'completed'
        return result

print('Experiment runner ready (cache-optimized)')

Experiment runner ready (cache-optimized)


In [ ]:
import signal

# ============================================================
# CHECKPOINT / RESUME SYSTEM
# ============================================================
CHECKPOINT_PATH = RESULTS_DIR / '_checkpoint.json'

def save_checkpoint(all_results, completed_names, phase, extra=None):
    """Atomic checkpoint save - crash-safe with verification."""
    ckpt = {
        'phase': phase,
        'completed_experiments': list(completed_names),
        'results_count': len(all_results),
        'results': all_results,
        'timestamp': datetime.now().isoformat(),
        'pid': os.getpid(),
    }
    if extra:
        ckpt.update(extra)
    tmp = CHECKPOINT_PATH.with_suffix('.tmp')
    with open(tmp, 'w') as f:
        json.dump(ckpt, f)
    try:
        with open(tmp, 'r') as f:
            verified = json.load(f)
        assert verified['phase'] == phase
        assert verified['results_count'] == len(all_results)
    except Exception as e:
        print(f'  WARNING: Checkpoint verification failed: {e}')
        tmp.unlink(missing_ok=True)
        return
    tmp.replace(CHECKPOINT_PATH)

def load_checkpoint():
    """Load checkpoint if it exists."""
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f:
                ckpt = json.load(f)
            if 'phase' not in ckpt or 'completed_experiments' not in ckpt:
                print(f'  WARNING: Checkpoint missing required fields, ignoring')
                return None
            return ckpt
        except Exception as e:
            print(f'  WARNING: Checkpoint corrupted ({e}), will re-run from scratch')
            backup = CHECKPOINT_PATH.with_suffix('.corrupted')
            try:
                CHECKPOINT_PATH.rename(backup)
                print(f'  Corrupted checkpoint backed up to: {backup}')
            except:
                pass
    return None

def aggressive_cleanup():
    """Free as much memory as possible."""
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except: pass
    try:
        # Try to release Python memory back to OS
        import ctypes
        libc = ctypes.CDLL("libc.so.6")
        libc.malloc_trim(0)
    except: pass

# ============================================================
# SIGNAL HANDLER - save checkpoint on kill/timeout
# ============================================================
_INTERRUPT_STATE = {'saving': False}

def _signal_handler(signum, frame):
    if _INTERRUPT_STATE['saving']:
        return  # already saving, don't recurse
    _INTERRUPT_STATE['saving'] = True
    sig_name = signal.Signals(signum).name
    print(f'\n\n!!! CAUGHT {sig_name} - saving checkpoint before exit !!!')
    try:
        if 'all_results' in dir() or 'all_results' in frame.f_locals:
            results = frame.f_locals.get('all_results', [])
            completed = [r.get('name', '?') for r in results if r.get('status') == 'completed']
            save_checkpoint(results, completed, 'interrupted',
                          {'interrupt_signal': sig_name})
            print(f'  Checkpoint saved: {len(completed)} experiments completed')
    except Exception as e:
        print(f'  Checkpoint save failed: {e}')
    _INTERRUPT_STATE['saving'] = False

signal.signal(signal.SIGTERM, _signal_handler)
signal.signal(signal.SIGINT, _signal_handler)

print('Checkpoint/resume system ready')
print(f'  Checkpoint file: {CHECKPOINT_PATH}')
print(f'  Signal handlers: SIGTERM, SIGINT')

# ============================================================
# DEFINE EXPERIMENTS
# ============================================================
EXPERIMENTS = [
    ExperimentRunner(
        name='ablation_0_full',
        description='Full baseline: TF-IDF + 117 handcrafted features',
        use_tfidf=True, use_handcrafted=True,
    ),
    ExperimentRunner(
        name='ablation_1_tfidf_only',
        description='TF-IDF features only, no handcrafted features',
        use_tfidf=True, use_handcrafted=False,
    ),
    ExperimentRunner(
        name='ablation_2_handcrafted_only',
        description='Handcrafted features only, no TF-IDF',
        use_tfidf=False, use_handcrafted=True,
    ),
    ExperimentRunner(
        name='ablation_3_no_text_stats',
        description='Full minus text_statistics group',
        use_tfidf=True, use_handcrafted=True,
        exclude_groups=['text_statistics'],
    ),
    ExperimentRunner(
        name='ablation_4_no_char_comp',
        description='Full minus char_composition group',
        use_tfidf=True, use_handcrafted=True,
        exclude_groups=['char_composition'],
    ),
    ExperimentRunner(
        name='ablation_5_no_entropy',
        description='Full minus entropy_diversity group',
        use_tfidf=True, use_handcrafted=True,
        exclude_groups=['entropy_diversity'],
    ),
    ExperimentRunner(
        name='ablation_6_no_keywords',
        description='Full minus keywords group',
        use_tfidf=True, use_handcrafted=True,
        exclude_groups=['keywords'],
    ),
    ExperimentRunner(
        name='ablation_7_no_encoding',
        description='Full minus encoding_patterns group',
        use_tfidf=True, use_handcrafted=True,
        exclude_groups=['encoding_patterns'],
    ),
    ExperimentRunner(
        name='ablation_8_no_structural',
        description='Full minus structural_patterns group',
        use_tfidf=True, use_handcrafted=True,
        exclude_groups=['structural_patterns'],
    ),
    ExperimentRunner(
        name='ablation_9_no_linguistic',
        description='Full minus linguistic group',
        use_tfidf=True, use_handcrafted=True,
        exclude_groups=['linguistic'],
    ),
    ExperimentRunner(
        name='ablation_10_no_attack',
        description='Full minus attack_specific group',
        use_tfidf=True, use_handcrafted=True,
        exclude_groups=['attack_specific'],
    ),
    ExperimentRunner(
        name='ablation_11_no_injection',
        description='Full minus injection_detection group',
        use_tfidf=True, use_handcrafted=True,
        exclude_groups=['injection_detection'],
    ),
    ExperimentRunner(
        name='ablation_12_no_readability',
        description='Full minus readability group',
        use_tfidf=True, use_handcrafted=True,
        exclude_groups=['readability'],
    ),
]

print(f'Defined {len(EXPERIMENTS)} ablation experiments:')
for i, exp in enumerate(EXPERIMENTS):
    print(f'  [{i:2d}] {exp.name}: {exp.description}')

# Feature cache persistence (survives kernel restart)
FEATURE_CACHE_PATH = INTERMEDIATE_DIR / 'feature_cache.pkl'
TFIDF_CACHE_PATH = INTERMEDIATE_DIR / 'tfidf_cache.pkl'

def save_feature_cache():
    if _FEATURE_CACHE['all_features'] is not None:
        joblib.dump({'all_features': _FEATURE_CACHE['all_features'],
                      'feature_names': _FEATURE_CACHE['feature_names']}, FEATURE_CACHE_PATH)
        print(f'  Feature cache saved to disk')
    if _FEATURE_CACHE['tfidf_full'] is not None:
        joblib.dump({'tfidf_full': _FEATURE_CACHE['tfidf_full'],
                      'tfidf_train': _FEATURE_CACHE['tfidf_train'],
                      'tfidf_val': _FEATURE_CACHE['tfidf_val'],
                      'tfidf_test': _FEATURE_CACHE['tfidf_test']}, TFIDF_CACHE_PATH)
        print(f'  TF-IDF cache saved to disk')

def load_feature_cache_from_disk():
    loaded = False
    if FEATURE_CACHE_PATH.exists():
        try:
            data = joblib.load(FEATURE_CACHE_PATH)
            _FEATURE_CACHE['all_features'] = data['all_features']
            _FEATURE_CACHE['feature_names'] = data['feature_names']
            print(f'  Feature cache loaded from disk')
            loaded = True
        except Exception as e:
            print(f'  WARNING: Failed to load feature cache: {e}')
    if TFIDF_CACHE_PATH.exists():
        try:
            data = joblib.load(TFIDF_CACHE_PATH)
            _FEATURE_CACHE['tfidf_full'] = data['tfidf_full']
            _FEATURE_CACHE['tfidf_train'] = data['tfidf_train']
            _FEATURE_CACHE['tfidf_val'] = data['tfidf_val']
            _FEATURE_CACHE['tfidf_test'] = data['tfidf_test']
            print(f'  TF-IDF cache loaded from disk')
            loaded = True
        except Exception as e:
            print(f'  WARNING: Failed to load TF-IDF cache: {e}')
    return loaded

# ============================================================
# EXECUTION
# ============================================================
all_results = []
print('\n' + '=' * 70)
print('ABLATION STUDY: Hybrid Guardrail V3')
print('=' * 70)

# Check for existing checkpoint (resume)
checkpoint = load_checkpoint()
completed_names = set()
if checkpoint and checkpoint.get('phase') == 'precompute_done':
    completed_names = set(checkpoint.get('completed_experiments', []))
    print(f'\nRESUMING from checkpoint: {len(completed_names)} experiments already done')

# ============================================================
# PHASE 1: Precompute features ONCE
# ============================================================
if checkpoint and checkpoint.get('phase') in ('precompute_done', 'experiments_done'):
    print('\nPHASE 1: Precomputation already done (from checkpoint)')
    cache_loaded = load_feature_cache_from_disk()
    if _FEATURE_CACHE['all_features'] is None:
        if not cache_loaded:
            print('  Feature cache empty and not on disk - re-precomputing...')
        precompute_all_features(Xtr, Xva, Xte)
        precompute_tfidf(Xtr, Xva, Xte)
        save_feature_cache()
        save_checkpoint([], list(completed_names), 'precompute_done')
else:
    print('\n' + '=' * 70)
    print('PHASE 1: Precomputation (one-time cost)')
    print('=' * 70)
    t_total = time.time()
    
    precompute_all_features(Xtr, Xva, Xte)
    precompute_tfidf(Xtr, Xva, Xte)
    save_feature_cache()
    
    precompute_time = time.time() - t_total
    print(f'\nPrecomputation done in {precompute_time:.1f}s')
    save_checkpoint([], list(completed_names), 'precompute_done')

print(f'Cached: {len(Xtr):,} train + {len(Xva):,} val + {len(Xte):,} test')
print(f'Features: {_FEATURE_CACHE["all_features"]["train"].shape[1]} HC + {_FEATURE_CACHE["tfidf_train"].shape[1]} TFIDF')

# ============================================================
# PHASE 2: Run experiments with checkpoint/resume
# ============================================================
print('\n' + '=' * 70)
print('PHASE 2: Experiments')
print('=' * 70)

t_total = time.time()
t_experiments = time.time()
MAX_RETRIES = 2

for i, exp in enumerate(EXPERIMENTS):
    # Skip already completed
    if exp.name in completed_names:
        result_path = RESULTS_DIR / f'{exp.name}_result.json'
        if result_path.exists():
            try:
                with open(result_path, 'r') as f:
                    existing = json.load(f)
                if existing.get('status') == 'completed':
                    required = ['accuracy', 'f1', 'roc_auc']
                    missing = [m for m in required if m not in existing or existing[m] is None]
                    if missing:
                        print(f'  [{i+1:2d}] {exp.name}: RESULT INCOMPLETE (missing {missing}), re-running')
                    else:
                        print(f'  [{i+1:2d}] {exp.name}: COMPLETED (skip)')
                        all_results.append(existing)
                        continue
            except Exception as e:
                print(f'  [{i+1:2d}] {exp.name}: RESULT CORRUPTED ({e}), re-running')
    
    print(f'\n{"=" * 70}')
    print(f'EXPERIMENT {i+1}/{len(EXPERIMENTS)}: {exp.name}')
    print(f'{"=" * 70}')
    
    update_manifest(exp.name, {
        'status': 'running',
        'start_time': datetime.now().isoformat(),
        'description': exp.description,
    })
    
    success = False
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            result = exp.run(Xtr, ytr, Xva, yva, Xte, yte, get_feature_names(), SEED)
            all_results.append(result)
            completed_names.add(exp.name)
            update_manifest(exp.name, {
                'status': 'completed',
                'end_time': datetime.now().isoformat(),
                'metrics': {k: result[k] for k in ['accuracy','f1','roc_auc','pr_auc','fpr','fnr'] if k in result},
            })
            # CHECKPOINT after every experiment
            save_checkpoint(all_results, list(completed_names), 'experiments_done')
            print(f'  -> COMPLETED ({time.time() - t_experiments:.1f}s elapsed)')
            success = True
            break
        except MemoryError as e:
            print(f'  -> OUT OF MEMORY (attempt {attempt}/{MAX_RETRIES}): {e}')
            aggressive_cleanup()
            if _FEATURE_CACHE['all_features'] is None:
                print(f'  -> Feature cache evicted, re-precomputing...')
                precompute_all_features(Xtr, Xva, Xte)
            if _FEATURE_CACHE['tfidf_full'] is None:
                print(f'  -> TF-IDF cache evicted, re-precomputing...')
                precompute_tfidf(Xtr, Xva, Xte)
            if attempt < MAX_RETRIES:
                print(f'  -> Retrying in 10s...')
                time.sleep(10)
        except Exception as e:
            print(f'  -> FAILED (attempt {attempt}/{MAX_RETRIES}): {e}')
            import traceback; traceback.print_exc()
            aggressive_cleanup()
            if attempt < MAX_RETRIES:
                print(f'  -> Retrying in 5s...')
                time.sleep(5)
    
    if not success:
        update_manifest(exp.name, {
            'status': 'failed',
            'error': f'Failed after {MAX_RETRIES} attempts',
            'end_time': datetime.now().isoformat(),
        })
        save_checkpoint(all_results, list(completed_names), 'experiments_done')
    
    # AGGRESSIVE cleanup between experiments
    aggressive_cleanup()
    time.sleep(1)  # Brief pause for OS memory reclamation

total_time = time.time() - t_total
print(f'\n{"=" * 70}')
print(f'ABLATION STUDY COMPLETE: {len(all_results)}/{len(EXPERIMENTS)} succeeded')
print(f'Total time: {total_time:.1f}s ({total_time/60:.1f}min)')
completed_results = [r for r in all_results if r.get('status') == 'completed']
if completed_results:
    avg_exp = (time.time() - t_experiments) / max(1, len(completed_results))
    print(f'Avg per experiment: {avg_exp:.1f}s')
print(f'{"=" * 70}')

Checkpoint/resume system ready
  Checkpoint file: /kaggle/working/ablation_study/results/_checkpoint.json
  Signal handlers: SIGTERM, SIGINT
Defined 13 ablation experiments:
  [ 0] ablation_0_full: Full baseline: TF-IDF + 117 handcrafted features
  [ 1] ablation_1_tfidf_only: TF-IDF features only, no handcrafted features
  [ 2] ablation_2_handcrafted_only: Handcrafted features only, no TF-IDF
  [ 3] ablation_3_no_text_stats: Full minus text_statistics group
  [ 4] ablation_4_no_char_comp: Full minus char_composition group
  [ 5] ablation_5_no_entropy: Full minus entropy_diversity group
  [ 6] ablation_6_no_keywords: Full minus keywords group
  [ 7] ablation_7_no_encoding: Full minus encoding_patterns group
  [ 8] ablation_8_no_structural: Full minus structural_patterns group
  [ 9] ablation_9_no_linguistic: Full minus linguistic group
  [10] ablation_10_no_attack: Full minus attack_specific group
  [11] ablation_11_no_injection: Full minus injection_detection group
  [12] ablation_12_n

[17:43:52] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


    Train: 0.7444, Val: 0.7452 (2273.8s)
  Calibrating...


[18:08:34] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[18:10:22] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booste

    ECE: 0.0161, Threshold: 0.46 (330.3s)
  Evaluating on test set...
    Inference: 10.97s (6588/s, 0.152ms/prompt)

    === ablation_0_full ===
              precision    recall  f1-score   support

        Safe       0.82      0.75      0.78     28978
   Malicious       0.84      0.89      0.86     43307

    accuracy                           0.83     72285
   macro avg       0.83      0.82      0.82     72285
weighted avg       0.83      0.83      0.83     72285

  -> FAILED (attempt 1/2): Invalid format specifier '.4f if au else "N/A"' for object of type 'float'


...

[22:27:16] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[22:54:48] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher 

    ECE: 0.0107, Threshold: 0.44 (320.3s)
  Evaluating on test set...
    Inference: 10.99s (6576/s, 0.152ms/prompt)

    === ablation_3_no_text_stats ===
              precision    recall  f1-score   support

        Safe       0.83      0.77      0.80     28978
   Malicious       0.85      0.90      0.87     43307

    accuracy                           0.85     72285
   macro avg       0.84      0.83      0.84     72285
weighted avg       0.84      0.85      0.84     72285

  -> FAILED (attempt 2/2): Invalid format specifier '.4f if au else "N/A"' for object of type 'float'


Traceback (most recent call last):
  File "/tmp/ipykernel_58/2120958813.py", line 313, in <cell line: 0>
    result = exp.run(Xtr, ytr, Xva, yva, Xte, yte, get_feature_names(), SEED)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_58/3528866180.py", line 236, in run
    print(f'    Acc={ac:.4f} F1={f1_:.4f} AUC={au:.4f if au else "N/A"} FPR={fpr_val:.4f} FNR={fnr_val:.4f}')
                                              ^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: Invalid format specifier '.4f if au else "N/A"' for object of type 'float'



EXPERIMENT 5/13: ablation_4_no_char_comp
    Features: 5111 (111 HC + 5000 TFIDF) in 3.402s
  Training stacking ensemble...
    XGBoost: GPU
    LightGBM: CPU
    RandomForest: CPU (all cores)


[23:09:21] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# Build summary table
rows = []
for r in all_results:
    if r.get('status') != 'completed':
        continue
    components = []
    removed = []
    if r.get('use_tfidf', True):
        components.append('TF-IDF')
    else:
        removed.append('TF-IDF')
    if r.get('use_handcrafted', True):
        components.append('Handcrafted')
    else:
        removed.append('Handcrafted')
    for g in r.get('exclude_groups', []):
        removed.append(g)
    rows.append({
        'experiment': r['name'],
        'components_used': ' + '.join(components) if components else 'None',
        'components_removed': ', '.join(removed) if removed else 'None',
        'train_samples': r.get('train_samples', 0),
        'val_samples': r.get('val_samples', 0),
        'test_samples': r.get('test_samples', 0),
        'total_features': r.get('total_features', 0),
        'accuracy': r.get('accuracy'),
        'precision': r.get('precision'),
        'recall': r.get('recall'),
        'f1': r.get('f1'),
        'roc_auc': r.get('roc_auc'),
        'pr_auc': r.get('pr_auc'),
        'fpr': r.get('fpr'),
        'fnr': r.get('fnr'),
        'inference_latency': r.get('inference_latency_ms'),
        'training_time': r.get('train_time'),
    })

if not rows:
    print('WARNING: No completed experiments found. Summary table cannot be generated.')
else:
    df_summary = pd.DataFrame(rows)
    csv_path = TABLE_DIR / 'ablation_summary.csv'
    df_summary.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')
    json_path = TABLE_DIR / 'ablation_summary.json'
    atomic_save_json(rows, json_path)
    print(f'Saved: {json_path}')
    display_cols = [c for c in ['experiment', 'total_features', 'accuracy', 'f1', 'roc_auc', 'pr_auc', 'fpr', 'fnr'] if c in df_summary.columns]
    print('\n' + '=' * 120)
    print('ABLATION SUMMARY TABLE')
    print('=' * 120)
    print(df_summary[display_cols].to_string(index=False))
    print('=' * 120)

In [ ]:
# Find baseline result
baseline = None
for r in all_results:
    if r.get('name') == 'ablation_0_full' and r.get('status') == 'completed':
        baseline = r
        break

completed = [r for r in all_results if r.get('status') == 'completed']

if baseline is None:
    print('WARNING: Baseline not found! Delta analysis skipped.')
elif len(completed) < 2:
    print('WARNING: Fewer than 2 completed experiments. Delta analysis limited.')
else:
    print('Delta Analysis (relative to full baseline)')
    print('=' * 80)
    delta_rows = []
    for r in completed:
        if r.get('name') == 'ablation_0_full':
            continue
        delta = {
            'experiment': r['name'],
            'delta_accuracy': r.get('accuracy', 0) - baseline.get('accuracy', 0),
            'delta_f1': r.get('f1', 0) - baseline.get('f1', 0),
            'delta_roc_auc': (r.get('roc_auc', 0) or 0) - (baseline.get('roc_auc', 0) or 0),
            'delta_pr_auc': (r.get('pr_auc', 0) or 0) - (baseline.get('pr_auc', 0) or 0),
            'delta_fpr': r.get('fpr', 0) - baseline.get('fpr', 0),
            'delta_fnr': r.get('fnr', 0) - baseline.get('fnr', 0),
        }
        delta_rows.append(delta)
    if delta_rows:
        df_delta = pd.DataFrame(delta_rows)
        delta_csv = TABLE_DIR / 'ablation_deltas.csv'
        df_delta.to_csv(delta_csv, index=False)
        atomic_save_json(delta_rows, TABLE_DIR / 'ablation_deltas.json')
        print(df_delta.to_string(index=False))
        print(f'\nSaved: {delta_csv}')
        print('\n' + '=' * 80)
        print('KEY FINDINGS')
        print('=' * 80)
        worst = min(delta_rows, key=lambda x: x['delta_f1'])
        print(f'  Largest F1 decrease: {worst["experiment"]} (dF1 = {worst["delta_f1"]:+.4f})')
        best_delta = max(delta_rows, key=lambda x: x['delta_f1'])
        print(f'  Smallest F1 decrease: {best_delta["experiment"]} (dF1 = {best_delta["delta_f1"]:+.4f})')
        tfidf_only = next((r for r in all_results if r.get('name') == 'ablation_1_tfidf_only'), None)
        hc_only = next((r for r in all_results if r.get('name') == 'ablation_2_handcrafted_only'), None)
        if tfidf_only and hc_only:
            print(f'  TF-IDF only F1: {tfidf_only.get("f1", "N/A")}')
            print(f'  Handcrafted only F1: {hc_only.get("f1", "N/A")}')
            if tfidf_only.get('f1', 0) > hc_only.get('f1', 0):
                print('  -> TF-IDF alone performs better than handcrafted features alone')
            else:
                print('  -> Handcrafted features alone perform better than TF-IDF alone')
    else:
        print('No ablation experiments completed for delta analysis.')

In [ ]:
print('Generating publication-quality figures...')

# Filter completed results
completed = [r for r in all_results if r.get('status') == 'completed']

if len(completed) < 1:
    print('WARNING: No completed experiments. Skipping figure generation.')
else:
    names = [r['name'].replace('ablation_', 'A').replace('_', ' ') for r in completed]
    accs = [r.get('accuracy', 0) for r in completed]
    f1s = [r.get('f1', 0) for r in completed]
    aucs = [r.get('roc_auc', 0) or 0 for r in completed]
    prs = [r.get('pr_auc', 0) or 0 for r in completed]
    fprs = [r.get('fpr', 0) for r in completed]
    fnrs = [r.get('fnr', 0) for r in completed]

    colors = ['#2196F3'] + ['#FF9800'] * (len(completed) - 1)

    # Figure 1: Performance comparison (grouped bar)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Hybrid Guardrail V3 — Ablation Study Results', fontsize=18, fontweight='bold', y=1.02)

    # Accuracy
    ax = axes[0, 0]
    bars = ax.bar(range(len(names)), accs, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title('Accuracy', fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(min(accs) - 0.05, max(accs) + 0.02)
    for bar, v in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7)
    ax.grid(axis='y', alpha=0.3)

    # F1
    ax = axes[0, 1]
    bars = ax.bar(range(len(names)), f1s, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_ylabel('F1 Score', fontsize=12)
    ax.set_title('F1 Score', fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(min(f1s) - 0.05, max(f1s) + 0.02)
    for bar, v in zip(bars, f1s):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7)
    ax.grid(axis='y', alpha=0.3)

    # ROC-AUC
    ax = axes[1, 0]
    bars = ax.bar(range(len(names)), aucs, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_ylabel('ROC-AUC', fontsize=12)
    ax.set_title('ROC-AUC', fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(min(aucs) - 0.05, max(aucs) + 0.02)
    for bar, v in zip(bars, aucs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7)
    ax.grid(axis='y', alpha=0.3)

    # FPR
    ax = axes[1, 1]
    bars = ax.bar(range(len(names)), fprs, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_ylabel('False Positive Rate', fontsize=12)
    ax.set_title('False Positive Rate (lower is better)', fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, max(fprs) + 0.02)
    for bar, v in zip(bars, fprs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7)
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'ablation_performance.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('  Saved: ablation_performance.png')

    # Figure 2: AUC comparison
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(names))
    w = 0.35
    ax.bar(x - w/2, aucs, w, label='ROC-AUC', color='#2196F3', edgecolor='white')
    ax.bar(x + w/2, prs, w, label='PR-AUC', color='#E91E63', edgecolor='white')
    ax.set_ylabel('AUC', fontsize=12)
    ax.set_title('ROC-AUC vs PR-AUC Across Ablations', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'ablation_auc.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('  Saved: ablation_auc.png')

    # Figure 3: F1 comparison
    fig, ax = plt.subplots(figsize=(14, 6))
    bars = ax.bar(range(len(names)), f1s, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_ylabel('F1 Score', fontsize=12)
    ax.set_title('F1 Score Across Ablations', fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    for bar, v in zip(bars, f1s):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'ablation_f1.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('  Saved: ablation_f1.png')

    # Figure 4: FPR comparison
    fig, ax = plt.subplots(figsize=(14, 6))
    bars = ax.bar(range(len(names)), fprs, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_ylabel('False Positive Rate', fontsize=12)
    ax.set_title('False Positive Rate Across Ablations (lower is better)', fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    for bar, v in zip(bars, fprs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'ablation_fpr.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('  Saved: ablation_fpr.png')

    # Figure 5: Performance degradation from baseline
    if baseline:
        fig, ax = plt.subplots(figsize=(14, 6))
        delta_names = [r['name'].replace('ablation_', 'A').replace('_', ' ') for r in delta_rows]
        delta_f1_vals = [d['delta_f1'] for d in delta_rows]
        delta_colors = ['#4CAF50' if v >= 0 else '#F44336' for v in delta_f1_vals]
        bars = ax.bar(range(len(delta_names)), delta_f1_vals, color=delta_colors, edgecolor='white')
        ax.axhline(y=0, color='black', linewidth=0.5)
        ax.set_ylabel('ΔF1 (relative to full baseline)', fontsize=12)
        ax.set_title('Performance Degradation Relative to Full Baseline', fontsize=14, fontweight='bold')
        ax.set_xticks(range(len(delta_names)))
        ax.set_xticklabels(delta_names, rotation=45, ha='right', fontsize=8)
        for bar, v in zip(bars, delta_f1_vals):
            y_pos = bar.get_height() if v >= 0 else bar.get_height() - 0.003
            ax.text(bar.get_x() + bar.get_width()/2, y_pos,
                    f'{v:+.3f}', ha='center', va='bottom' if v >= 0 else 'top', fontsize=8)
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'ablation_degradation.png', dpi=300, bbox_inches='tight')
        plt.close()
        print('  Saved: ablation_degradation.png')

    # Figure 6: Confusion matrices (grid)
    n_exp = len(completed)
    ncols = min(4, n_exp)
    nrows = (n_exp + ncols - 1) // ncols
    fig, axes_grid = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
    if nrows == 1 and ncols == 1:
        axes_grid = np.array([[axes_grid]])
    elif nrows == 1:
        axes_grid = axes_grid.reshape(1, -1)
    elif ncols == 1:
        axes_grid = axes_grid.reshape(-1, 1)

    for idx, r in enumerate(completed):
        row, col = divmod(idx, ncols)
        ax = axes_grid[row, col]
        cm_path = RESULTS_DIR / f'{r["name"]}_confusion.json'
        if cm_path.exists():
            with open(cm_path) as f:
                cm_data = json.load(f)
            cm_ = np.array(cm_data['confusion_matrix'])
            sns.heatmap(cm_, annot=True, fmt='d', cmap='Blues',
                        xticklabels=['Safe', 'Malicious'], yticklabels=['Safe', 'Malicious'], ax=ax)
        ax.set_title(r['name'].replace('ablation_', 'A').replace('_', ' '), fontsize=10)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')

    # Hide unused subplots
    for idx in range(n_exp, nrows * ncols):
        row, col = divmod(idx, ncols)
        axes_grid[row, col].set_visible(False)

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'ablation_confusion_matrices.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('  Saved: ablation_confusion_matrices.png')

    # Figure 7: ROC curves overlay
    fig, ax = plt.subplots(figsize=(10, 8))
    for r in completed:
        roc_path = RESULTS_DIR / f'{r["name"]}_roc.json'
        if roc_path.exists():
            with open(roc_path) as f:
                roc_data = json.load(f)
            label = r['name'].replace('ablation_', 'A').replace('_', ' ')
            ax.plot(roc_data['fpr'], roc_data['tpr'], linewidth=1.5,
                    label=f'{label} (AUC={roc_data["auc"]:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curves — All Ablations', fontsize=14, fontweight='bold')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'ablation_roc_curves.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('  Saved: ablation_roc_curves.png')

    # Figure 8: PR curves overlay
    fig, ax = plt.subplots(figsize=(10, 8))
    for r in completed:
        pr_path = RESULTS_DIR / f'{r["name"]}_pr.json'
        if pr_path.exists():
            with open(pr_path) as f:
                pr_data = json.load(f)
            label = r['name'].replace('ablation_', 'A').replace('_', ' ')
            ax.plot(pr_data['recall'], pr_data['precision'], linewidth=1.5,
                    label=f'{label} (AP={pr_data["ap"]:.4f})')
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title('Precision-Recall Curves — All Ablations', fontsize=14, fontweight='bold')
    ax.legend(fontsize=7, loc='lower left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'ablation_pr_curves.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('  Saved: ablation_pr_curves.png')

    print('All figures generated.')

In [ ]:
# Generate LaTeX table
def generate_latex(results, baseline_r):
    lines = []
    lines.append('\\begin{table*}[t]')
    lines.append('\\centering')
    lines.append('\\caption{Ablation Study Results for Hybrid Guardrail V3. Bold values indicate the best performance in each column.}')
    lines.append('\\label{tab:ablation}')
    lines.append('\\small')
    lines.append('\\begin{tabular}{l c c c c c c c c c}')
    lines.append('\\toprule')
    lines.append('\\textbf{Experiment} & \\textbf{Feat.} & \\textbf{Acc.} & \\textbf{Prec.} & \\textbf{Rec.} & \\textbf{F1} & \\textbf{ROC-AUC} & \\textbf{PR-AUC} & \\textbf{FPR} & \\textbf{FNR} \\\\')
    lines.append('\\midrule')
    completed = [r for r in results if r.get('status') == 'completed']
    best = {}
    for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']:
        vals = [(r.get(metric, 0) or 0) for r in completed]
        best[metric] = max(vals) if vals else 0
    best['fpr'] = min((r.get('fpr', 1) or 1) for r in completed) if completed else 0
    best['fnr'] = min((r.get('fnr', 1) or 1) for r in completed) if completed else 0
    def fmt(val, metric, bold=False):
        if val is None:
            return 'N/A'
        s = f'{val:.4f}'
        if bold and abs(val - best.get(metric, 0)) < 1e-6:
            return f'\\textbf{{{s}}}'
        return s
    for r in completed:
        name = r['name'].replace('ablation_', '').replace('_', ' ')
        nf = r.get('total_features', 0)
        is_baseline = (r.get('name') == 'ablation_0_full')
        delta_str = ''
        if not is_baseline and baseline_r:
            d_f1 = r.get('f1', 0) - baseline_r.get('f1', 0)
            delta_str = f' (dF1={d_f1:+.3f})' if abs(d_f1) > 1e-6 else ''
        row = (
            f'{name}{delta_str} & {nf} '
            f'& {fmt(r.get("accuracy"), "accuracy", not is_baseline)} '
            f'& {fmt(r.get("precision"), "precision", not is_baseline)} '
            f'& {fmt(r.get("recall"), "recall", not is_baseline)} '
            f'& {fmt(r.get("f1"), "f1", not is_baseline)} '
            f'& {fmt(r.get("roc_auc"), "roc_auc", not is_baseline)} '
            f'& {fmt(r.get("pr_auc"), "pr_auc", not is_baseline)} '
            f'& {fmt(r.get("fpr"), "fpr", not is_baseline)} '
            f'& {fmt(r.get("fnr"), "fnr", not is_baseline)} '
            f'\\\\'
        )
        lines.append(row)
    lines.append('\\bottomrule')
    lines.append('\\end{tabular}')
    lines.append('\\end{table*}')
    return '\n'.join(lines)

completed = [r for r in all_results if r.get('status') == 'completed']
if not completed:
    print('WARNING: No completed experiments. LaTeX table cannot be generated.')
else:
    latex = generate_latex(all_results, baseline)
    tex_path = TABLE_DIR / 'ablation_results.tex'
    with open(tex_path, 'w') as f:
        f.write(latex)
    print(f'Saved: {tex_path}')
    print('\nLaTeX table:')
    print(latex)

## Research Interpretation

The following auto-generated summary interprets the ablation results under the evaluated setting.

In [ ]:
print('=' * 80)
print('RESEARCH INTERPRETATION')
print('=' * 80)

completed = [r for r in all_results if r.get('status') == 'completed']

if baseline is None:
    print('WARNING: Baseline not available. Interpretation skipped.')
elif len(completed) < 2:
    print('WARNING: Fewer than 2 completed experiments. Limited interpretation.')
else:
    if delta_rows:
        worst = min(delta_rows, key=lambda x: x['delta_f1'])
        print('\n1. Largest F1 decrease from removing a component:')
        print(f'   {worst["experiment"]} produced a dF1 of {worst["delta_f1"]:+.4f}.')
        print('   This suggests that the removed component was associated with meaningful performance under the evaluated setting.')
    if delta_rows:
        worst = min(delta_rows, key=lambda x: x['delta_f1'])
        print('\n2. Most important feature group:')
        print(f'   {worst["experiment"]} (dF1 = {worst["delta_f1"]:+.4f}) appears to be the most important feature group.')
    if delta_rows:
        best_delta = max(delta_rows, key=lambda x: x['delta_f1'])
        print('\n3. Least important feature group:')
        print(f'   {best_delta["experiment"]} (dF1 = {best_delta["delta_f1"]:+.4f}) appears to contribute little to overall performance.')
    tfidf_only = next((r for r in all_results if r.get('name') == 'ablation_1_tfidf_only'), None)
    if tfidf_only and baseline:
        d = tfidf_only.get('f1', 0) - baseline.get('f1', 0)
        print('\n4. TF-IDF contribution:')
        print(f'   TF-IDF alone achieved F1={tfidf_only.get("f1", 0):.4f} (dF1 = {d:+.4f} from full).')
        if d > -0.05:
            print('   Under the evaluated setting, TF-IDF appears to be the dominant signal.')
        else:
            print('   Under the evaluated setting, TF-IDF contributes meaningfully but not dominantly.')
    hc_only = next((r for r in all_results if r.get('name') == 'ablation_2_handcrafted_only'), None)
    if hc_only and baseline:
        d = hc_only.get('f1', 0) - baseline.get('f1', 0)
        print('\n5. Handcrafted features contribution:')
        print(f'   Handcrafted features alone achieved F1={hc_only.get("f1", 0):.4f} (dF1 = {d:+.4f} from full).')
        if d > -0.05:
            print('   Under the evaluated setting, handcrafted features appear to be a strong signal.')
        else:
            print('   Under the evaluated setting, handcrafted features contribute meaningfully.')
    if tfidf_only and hc_only and baseline:
        bf1 = baseline.get('f1', 0)
        tf1 = tfidf_only.get('f1', 0)
        hf1 = hc_only.get('f1', 0)
        print('\n6. Hybrid vs. simpler variants:')
        print(f'   Full hybrid F1: {bf1:.4f}')
        print(f'   TF-IDF only F1: {tf1:.4f}')
        print(f'   Handcrafted only F1: {hf1:.4f}')
        if bf1 > max(tf1, hf1):
            print(f'   The full hybrid representation outperformed both simpler variants by {bf1 - max(tf1, hf1):.4f} F1 points.')
        elif bf1 < max(tf1, hf1):
            print('   Under the evaluated setting, the full hybrid did not outperform the best single-modality variant.')
        else:
            print('   The full hybrid performed comparably to the best single-modality variant.')
    if completed:
        scored = [(r, r.get('f1', 0) - 0.0001 * r.get('total_features', 0)) for r in completed]
        scored.sort(key=lambda x: x[1], reverse=True)
        best_tradeoff = scored[0][0]
        print('\n7. Best performance/complexity tradeoff:')
        print(f'   {best_tradeoff["name"]} with {best_tradeoff.get("total_features", 0)} features and F1={best_tradeoff.get("f1", 0):.4f}')
        print('   (under a simple linear complexity penalty of 0.0001 per feature)')

print('\n' + '=' * 80)
print('NOTE: All interpretations are under the evaluated setting. Interaction')
print('effects between feature groups may exist. A component that shows')
print('little individual impact may still contribute through interactions.')
print('=' * 80)

In [ ]:
# Save final manifest with all results
manifest = load_manifest()
manifest['study_completed'] = True
manifest['study_end_time'] = datetime.now().isoformat()
manifest['total_experiments'] = len(EXPERIMENTS)
manifest['completed_experiments'] = len([r for r in all_results if r.get('status') == 'completed'])
manifest['failed_experiments'] = len([r for r in all_results if r.get('status') == 'failed'])
manifest['dataset'] = DATASET_PATH
manifest['seed'] = SEED
manifest['split'] = '70/10/20 stratified'
manifest['results_dir'] = str(RESULTS_DIR)
manifest['figures_dir'] = str(FIG_DIR)
manifest['tables_dir'] = str(TABLE_DIR)
save_manifest(manifest)

print('Final manifest saved.')
print(f'\nAll outputs in: {ABLATION_DIR}')
print(f'  Results:   {RESULTS_DIR}')
print(f'  Figures:   {FIG_DIR}')
print(f'  Tables:    {TABLE_DIR}')
print(f'  Manifests: {MANIFEST_DIR}')
print('\nABLATION STUDY COMPLETE.')

In [ ]:
# ============================================================
# PUSH RESULTS AS KAGGLE DATASET
# ============================================================
import shutil
import subprocess
from pathlib import Path

print('Preparing results for Kaggle dataset upload...')

# 1. Create a staging directory
STAGING = Path('/kaggle/working/ablation_results_upload')
if STAGING.exists():
    shutil.rmtree(STAGING)
STAGING.mkdir(parents=True)

# 2. Copy all results into the staging directory
for src_dir in [RESULTS_DIR, FIG_DIR, TABLE_DIR, MANIFEST_DIR]:
    if src_dir.exists():
        dest = STAGING / src_dir.name
        shutil.copytree(src_dir, dest, dirs_exist_ok=True)
        print(f'  Copied {src_dir.name}/ ({len(list(dest.iterdir()))} files)')

# Also save a combined summary JSON
summary = {
    'study': 'Hybrid Guardrail V3 - Focused Ablation Study',
    'dataset': str(DATASET_PATH),
    'seed': SEED,
    'total_experiments': len(EXPERIMENTS),
    'completed': len([r for r in all_results if r.get('status') == 'completed']),
    'failed': len([r for r in all_results if r.get('status') == 'failed']),
    'experiments': [],
}
for r in all_results:
    summary['experiments'].append({
        'name': r.get('name'),
        'status': r.get('status'),
        'accuracy': r.get('accuracy'),
        'f1': r.get('f1'),
        'roc_auc': r.get('roc_auc'),
        'pr_auc': r.get('pr_auc'),
        'fpr': r.get('fpr'),
        'fnr': r.get('fnr'),
        'total_features': r.get('total_features'),
    })
atomic_save_json(summary, STAGING / 'study_summary.json')
print(f'  Saved study_summary.json')

# 3. Create dataset metadata
DATASET_SLUG = 'prashannadeveloper/guardrailer-ablation-results'
meta = {
    'title': 'Guardrailer V3 Ablation Study Results',
    'id': DATASET_SLUG,
    'licenses': [{'name': 'CC0-1.0'}],
}
meta_path = STAGING / 'dataset-metadata.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'  Created dataset-metadata.json')

# 4. Push to Kaggle
print(f'\nPushing to Kaggle as: {DATASET_SLUG}')
try:
    result = subprocess.run(
        ['kaggle', 'datasets', 'create', '-p', str(STAGING), '--dir-mode', 'zip'],
        capture_output=True, text=True, timeout=300
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f'\nKaggle API error (exit {result.returncode}):')
        print(result.stderr)
        print('\nFalling back to local ZIP download...')
        raise RuntimeError('kaggle CLI failed')
    else:
        print(f'\nResults published: https://www.kaggle.com/datasets/{DATASET_SLUG}')
except Exception as e:
    print(f'Kaggle push failed: {e}')
    # Fallback: create a downloadable ZIP in /kaggle/working
    zip_path = Path('/kaggle/working/ablation_results.zip')
    shutil.make_archive(str(zip_path).replace('.zip',''), 'zip', STAGING)
    print(f'\nFallback ZIP created: {zip_path}')
    print('Download it from the Kaggle notebook output panel (right side).')

# 5. Cleanup staging
shutil.rmtree(STAGING, ignore_errors=True)
print('\nDone!')
